# Bronze: shipments + inventory (Auto Loader, CSV)
Schema evolution: `carrier_code` appears in shipments from 2015-08-11; the stream stops once and resumes with the new column.

In [0]:
%run ./_common

In [0]:
NB = "11_bronze_csv"

def ingest_csv(name):
    tgt, cp = f"{CAT}.bronze.{name}", f"{CKPT}/bronze_{name}"
    def run():
        before = table_count(tgt)
        q = (spark.readStream.format("cloudFiles")
               .option("cloudFiles.format", "csv")
               .option("header", "true")
               .option("cloudFiles.schemaLocation", f"{cp}/schema")
               .option("cloudFiles.inferColumnTypes", "true")
               .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
               .option("rescuedDataColumn", "_rescued_data")
               .load(f"{LANDING}/{name}/")
               .select("*",
                       F.col("_metadata.file_path").alias("_source_file"),
                       F.current_timestamp().alias("_ingested_at"))
               .writeStream
               .option("checkpointLocation", cp)
               .option("mergeSchema", "true")
               .trigger(availableNow=True)
               .toTable(tgt))
        q.awaitTermination()
        print(f"{name}: {table_count(tgt) - before:,} rows ingested (total {table_count(tgt):,})")
    with_schema_retry(run)

for name in ["shipments", "inventory"]:
    ingest_csv(name)

In [0]:
for name in ["shipments", "inventory"]:
    r = spark.table(f"{CAT}.bronze.{name}").where("_rescued_data IS NOT NULL").count()
    dq_check(f"{name}_rescued_rows", r, r == 0, "0")

# every inventory snapshot day must contain the same number of rows (one per store x item)
sizes = spark.sql(f"SELECT COUNT(DISTINCT n) FROM (SELECT snapshot_date, COUNT(*) n FROM {CAT}.bronze.inventory GROUP BY 1)").first()[0]
dq_check("inventory_rows_per_day_constant", sizes, sizes == 1, "1 distinct size")
dq_finish(NB)